# CDCR Facility Heat Risk Index — v0.2

Computes a facility-level heat risk index for 31 CDCR state prisons following the Ovienmhada (2024) / VCP environmental risk framework.

**Risk = 0.25H + 0.25E + 0.50V** (additive, vulnerability double-weighted)

All sub-components are min-max normalized 0–1 before averaging within each component (the hazard temperature indicators use max-normalization — see §3). Components are combined additively with weights 0.25 (Hazard), 0.25 (Exposure), 0.50 (Vulnerability), then normalized 0–100 cross-period (current and mid-century share the same normalization denominator).

Vulnerability receives double weight because cooling in prisons is controlled by staff who, as Brunn et al. (2025) document, withhold AC, water, and shade to punish and retaliate. The index answers "where are people most at risk if cooling fails?" The additive form also prevents a facility with full mechanical AC from scoring zero risk; a multiplicative model would zero out CHCF, which has the highest vulnerability in the system but no indoor heat days.

**Version:** v0.2 — the hazard component was rebuilt on facility-relative temperature thresholds from a LOCA2-CA daily extraction, joined per facility rather than by tract centroid. Exposure, vulnerability, weights, periods, and cross-period normalization are unchanged from v0.1. See the changelog in `analysis/README.md`.

## Components

| Component | Weight | Sub-components | Source |
|---|---|---|---|
| **Hazard** | 0.25 | Hot days (`loca2_days_over_avg_plus10`) + warm nights (`loca2_nights_over_p95`), max-normalized and averaged, × AQI modifier (1 + 0.30·AQI_norm/100) | `data/hazards/heat_air_hazard.csv` via `cdcr_code` |
| **Exposure** | 0.25 | `days_indoor_above_78f_2025`, `ratio_indoor_to_outdoor`, `uhi_normalized`, `1 - pct_units_refrigeration` | `data/cdcr/indoor_outdoor_heat_2025.csv` + `data/cdcr/cdcr_facilities.csv` |
| **Vulnerability** | 0.50 | Medical acuity (P1+P2+medium), age >50, mental health (EOP), disability (DPP), race/POC, % female | `data/cdcr/cdcr_facilities.csv` |

**Note on facility coverage:** 31 of 34 state prisons have indoor exposure data. CAC, CVSP, and FWF are excluded (no indoor/outdoor heat model data available).

**Note on UHI nulls:** CCI and PVSP have no Benz & Burney (2021) UHI data — their tracts were classified as undeveloped. Imputed with system mean across 31 facilities.

**Note on PBSP ratio outlier:** PBSP (Pelican Bay, Crescent City coast) has `ratio_indoor_to_outdoor` = 15.75, driven by very few outdoor 78°F days (~4) in that coastal climate. This is physically plausible but will score PBSP at 1.0 on this sub-component. Flagged in output.

In [1]:
import pandas as pd
import numpy as np

# Load data
cdcr = pd.read_csv('data/cdcr/cdcr_facilities.csv')
hazard = pd.read_csv('data/hazards/heat_air_hazard.csv')
indoor = pd.read_csv('data/cdcr/indoor_outdoor_heat_2025.csv')

print(f'cdcr_facilities rows: {len(cdcr)}')
print(f'heat_air_hazard rows: {len(hazard)}')
print(f'indoor_outdoor_heat rows: {len(indoor)}')

cdcr_facilities rows: 84
heat_air_hazard rows: 357
indoor_outdoor_heat rows: 31


## 1. Build working dataset — 31 CDCR state prisons

In [2]:
# Filter to CDCR state prisons (has cdcr_code, not fire camp)
state_prisons = cdcr[
    cdcr['cdcr_code'].notna() &
    (cdcr['cdcr_firecamp'].fillna(False) != True)
].copy()
print(f'State prisons in cdcr_facilities: {len(state_prisons)}')

# Inner join with indoor_outdoor — this restricts to the 31 with exposure data
# (excludes CAC, CVSP, FWF which have no indoor heat model data)
df = state_prisons.merge(indoor, on='cdcr_code', how='inner', suffixes=('', '_indoor'))
print(f'After join with indoor_outdoor: {len(df)} facilities')
print(f'Facilities: {sorted(df["cdcr_code"].tolist())}')

State prisons in cdcr_facilities: 34
After join with indoor_outdoor: 31 facilities
Facilities: ['ASP', 'CAL', 'CCI', 'CCWF', 'CEN', 'CHCF', 'CIM', 'CIW', 'CMC', 'CMF', 'COR', 'CRC', 'CTF', 'FOL', 'HDSP', 'ISP', 'KVSP', 'LAC', 'MCSP', 'NKSP', 'PBSP', 'PVSP', 'RJD', 'SAC', 'SATF', 'SCC', 'SOL', 'SQ', 'SVSP', 'VSP', 'WSP']


In [3]:
# Join hazard by cdcr_code — v0.2: heat_air_hazard.csv is facility-keyed, so each
# prison gets its own LOCA2-CA cell (replacing the v0.1 tract-centroid join).
# Bring the relative-threshold counts and AQI; the hazard composite is recomputed
# below, max-normalized across these 31 index facilities.
haz_cols = [
    'cdcr_code',
    'loca2_days_over_avg_plus10_historic', 'loca2_days_over_avg_plus10_midcentury',
    'loca2_nights_over_p95_historic', 'loca2_nights_over_p95_midcentury',
    'AQI_norm',
]
df = df.merge(hazard[haz_cols], on='cdcr_code', how='left')

n_null = df['loca2_days_over_avg_plus10_historic'].isnull().sum()
print(f'Hazard join nulls: {n_null}')
print(f'days_over_avg_plus10 midcentury range: '
      f'{df["loca2_days_over_avg_plus10_midcentury"].min():.1f} – '
      f'{df["loca2_days_over_avg_plus10_midcentury"].max():.1f}')
print(f'nights_over_p95 midcentury range: '
      f'{df["loca2_nights_over_p95_midcentury"].min():.1f} – '
      f'{df["loca2_nights_over_p95_midcentury"].max():.1f}')
print(f'AQI_norm range: {df["AQI_norm"].min():.1f} – {df["AQI_norm"].max():.1f}')

Hazard join nulls: 0
days_over_avg_plus10 midcentury range: 14.0 – 44.3
nights_over_p95 midcentury range: 29.5 – 51.8
AQI_norm range: 12.2 – 94.4


## 2. Normalization helper

In [4]:
def minmax_norm(series):
    """Min-max normalize a series to 0–1 across the 31 facilities."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

## 3. Hazard component (v0.2)

Recomputed here from the facility-level LOCA2-CA counts in `data/hazards/heat_air_hazard.csv`,
max-normalized across these 31 index facilities so the hazard shares the same normalization base
as Exposure and Vulnerability:

1. **Hot days** — `loca2_days_over_avg_plus10`: days above the facility's own mean summer daily
   max + 10°F (Skarha threshold, 1981–2010 baseline).
2. **Warm nights** — `loca2_nights_over_p95`: nights (Apr–Oct) with tmin above the 95th
   percentile of the facility's 1961–1990 April–October minimum-temperature distribution
   (OEHHA convention).
3. Each is **max-normalized** across the 31 facilities and both periods, then averaged →
   `temp`. Max-normalization (not min-max) keeps a true zero: 0 means *no heat*, not *coldest of
   the 31*, so the coolest facility keeps its real small score.
4. Air quality enters as a **multiplicative modifier**, not a separate component:
   `H = temp × (1 + 0.30 × AQI_norm/100)`. It amplifies heat where both are present but cannot
   create hazard from pollution alone (`1 +` gives ×1.0 at AQI = 0). AQI_norm is the
   CalEnviroScreen 5.0 ozone/PM2.5/diesel percentile mean; missing AQI → ×1.
5. Scaled by the cross-period max of `H` to a 0–1 component score.

Both temperature indicators are relative thresholds — a change from v0.1, where daytime heat
used an absolute 90°F count (Cal-Adapt) and nighttime used a VCP relative threshold. See the v0.2
changelog in `analysis/README.md`.

In [5]:
# v0.2 hazard equation, normalized across the 31 index facilities (cross-period):
#   1. day  = loca2_days_over_avg_plus10   (hot days: facility mean summer tmax +10°F)
#      night = loca2_nights_over_p95        (warm nights: tmin > P95 of Apr–Oct, 1961–1990)
#   2. max-normalize each across the 31 facilities & both periods
#   3. temp = (day_norm + night_norm) / 2
#   4. H = temp × (1 + β·AQI_norm/100)      (AQI multiplicative modifier; missing → ×1)
#   5. scale by cross-period max of H  → hazard component on a 0–1 scale (like E and V)
BETA = 0.30  # AQI amplification coefficient — a design parameter, capped at +30%

day_max = df[['loca2_days_over_avg_plus10_historic',
              'loca2_days_over_avg_plus10_midcentury']].to_numpy().max()
night_max = df[['loca2_nights_over_p95_historic',
                'loca2_nights_over_p95_midcentury']].to_numpy().max()

day_norm_h = df['loca2_days_over_avg_plus10_historic']   / day_max
day_norm_m = df['loca2_days_over_avg_plus10_midcentury'] / day_max
night_norm_h = df['loca2_nights_over_p95_historic']   / night_max
night_norm_m = df['loca2_nights_over_p95_midcentury'] / night_max

temp_h = (day_norm_h + night_norm_h) / 2
temp_m = (day_norm_m + night_norm_m) / 2

# AQI multiplicative modifier (missing AQI → ×1)
modifier = 1 + BETA * (df['AQI_norm'].fillna(0) / 100)
H_h = temp_h * modifier
H_m = temp_m * modifier

# Scale by the cross-period max so the coldest facility keeps its real (non-zero)
# score — max-norm, not min-max: 0 means "no heat", not "coldest of the 31".
H_max = pd.concat([H_h, H_m]).max()
df['hazard_current']    = H_h / H_max
df['hazard_midcentury'] = H_m / H_max

print('Hazard component (0–1), v0.2 relative thresholds + AQI modifier (β=0.30):')
print(df[['cdcr_code', 'hazard_current', 'hazard_midcentury']]
      .sort_values('hazard_midcentury', ascending=False).round(3).to_string(index=False))

Hazard component (0–1), v0.2 relative thresholds + AQI modifier (β=0.30):
cdcr_code  hazard_current  hazard_midcentury
      CIM           0.327              1.000
      CIW           0.319              0.954
      CRC           0.279              0.889
     HDSP           0.237              0.834
      CCI           0.222              0.828
     CHCF           0.249              0.803
      LAC           0.223              0.788
       SQ           0.283              0.784
     CCWF           0.212              0.783
      VSP           0.212              0.777
     SATF           0.204              0.765
      COR           0.204              0.765
      SOL           0.304              0.760
      CMF           0.304              0.760
      FOL           0.262              0.759
      CMC           0.371              0.750
     MCSP           0.239              0.747
      RJD           0.306              0.741
      SCC           0.216              0.729
     NKSP           0.200 

## 4. Exposure component

4 equal-weight sub-components, each min-max normalized 0–1:
1. `days_indoor_above_78f_2025` — direct indoor heat burden
2. `ratio_indoor_to_outdoor` — building thermal amplification
3. `uhi_normalized` — geographic urban heat island (Benz & Burney 2021)
4. `1 - pct_units_refrigeration` — inverted AC coverage (high AC = low exposure)

In [6]:
# Sub-component 1: indoor 78°F days
df['exp_indoor78'] = minmax_norm(df['days_indoor_above_78f_2025'])

# Sub-component 2: ratio indoor/outdoor
# PBSP outlier: ratio = 15.75 vs system max ~2.0 for all others
print('ratio_indoor_to_outdoor — top 5:')
print(df[['cdcr_code', 'ratio_indoor_to_outdoor']]
      .sort_values('ratio_indoor_to_outdoor', ascending=False).head(5).to_string(index=False))
df['exp_ratio'] = minmax_norm(df['ratio_indoor_to_outdoor'])

# Sub-component 3: UHI (already 0–1; impute 2 nulls with system mean)
uhi_nulls = df.loc[df['uhi_normalized'].isnull(), 'cdcr_code'].tolist()
print(f'\nuhi_normalized nulls: {uhi_nulls} — imputed with system mean')
uhi_mean = df['uhi_normalized'].mean()
df['uhi_filled'] = df['uhi_normalized'].fillna(uhi_mean)
df['exp_uhi'] = minmax_norm(df['uhi_filled'])

# Sub-component 4: inverted AC fraction
# pct_units_refrigeration is 0–1; invert so high AC = low exposure
df['ac_inverted'] = 1 - df['pct_units_refrigeration']
df['exp_noac'] = minmax_norm(df['ac_inverted'])

# Exposure score = equal-weight mean of 4 sub-components
exp_cols = ['exp_indoor78', 'exp_ratio', 'exp_uhi', 'exp_noac']
df['exposure_score'] = df[exp_cols].mean(axis=1)

print('\nExposure sub-components and score:')
print(df[['cdcr_code'] + exp_cols + ['exposure_score']]
      .sort_values('exposure_score', ascending=False).to_string(index=False))

ratio_indoor_to_outdoor — top 5:
cdcr_code  ratio_indoor_to_outdoor
     PBSP                   15.750
      CTF                    1.935
      RJD                    1.932
       SQ                    1.414
      CCI                    1.278

uhi_normalized nulls: ['PVSP', 'CCI'] — imputed with system mean

Exposure sub-components and score:
cdcr_code  exp_indoor78  exp_ratio  exp_uhi  exp_noac  exposure_score
      COR      0.987578   0.061905 0.577900    1.0000        0.656846
     PBSP      0.391304   1.000000 0.309600    0.9048        0.651426
      SOL      0.987578   0.069143 0.581500    0.8657        0.625980
     NKSP      0.832298   0.052508 0.627600    0.9630        0.618852
     SATF      0.826087   0.051810 0.600800    0.9506        0.607324
      WSP      0.844720   0.052000 0.483900    0.9848        0.591355
      CIW      0.931677   0.058095 0.605100    0.5600        0.538718
      CIM      1.000000   0.063111 1.000000    0.0000        0.515778
      CCI      0.714286  

## 5. Vulnerability component

6 equal-weight sub-components, each min-max normalized 0–1:
1. **Medical acuity** — sum of P1 + P2 + medium CCHCS risk tiers (% of facility population)
2. **Age** — % over 50
3. **Mental health** — % EOP designation
4. **Disability** — % DPP placement
5. **Race/POC** — % people of color (heat inequity + structural vulnerability)
6. **Restricted housing (RHU)** — 12-month average % of facility population in restricted housing units (2025). Unlike the other sub-indicators, this variable captures constrained adaptive capacity rather than physiological susceptibility: RHU residents cannot access cooler areas of the facility, cannot self-regulate their location or activity during heat events, and have out-of-cell time averaging approximately one hour per day. Included in the vulnerability component because adaptive capacity is not treated as a standalone fourth component in this framework; RHU placement is the within-system structural condition that most acutely limits adaptive behavior. Source: CDCR Office of Research, STA429 Monthly Restricted Housing Reports, 2025. See Cloud et al. (2023) for the explicit link between solitary confinement and heat vulnerability.


In [7]:
# Medical acuity = P1 + P2 + medium risk
df['medical_acuity'] = (
    df['cchcs_high_risk_p1_pct_2025'] +
    df['cchcs_high_risk_p2_pct_2025'] +
    df['cchcs_medium_risk_pct_2025']
)

vuln_inputs = {
    'medical_acuity': 'medical_acuity',
    'age_over_50':    'cchcs_age_over_50_pct_2025',
    'mental_health':  'cchcs_mental_health_eop_pct_2025',
    'disability':     'cchcs_dpp_pct_2025',
    'race_poc':       'race_peopleofcolor_pct',
    'gender_female': 'gender_female_pct',
}

# Check for nulls
print('Vulnerability input nulls:')
for label, col in vuln_inputs.items():
    n = df[col].isnull().sum()
    print(f'  {label} ({col}): {n} nulls')

# Normalize each sub-component
vuln_norm_cols = []
for label, col in vuln_inputs.items():
    norm_col = f'vuln_{label}'
    df[norm_col] = minmax_norm(df[col])
    vuln_norm_cols.append(norm_col)

# Vulnerability score = equal-weight mean
df['vulnerability_score'] = df[vuln_norm_cols].mean(axis=1)

print('\nVulnerability sub-components and score:')
print(df[['cdcr_code'] + vuln_norm_cols + ['vulnerability_score']]
      .sort_values('vulnerability_score', ascending=False).to_string(index=False))

Vulnerability input nulls:
  medical_acuity (medical_acuity): 0 nulls
  age_over_50 (cchcs_age_over_50_pct_2025): 0 nulls
  mental_health (cchcs_mental_health_eop_pct_2025): 0 nulls
  disability (cchcs_dpp_pct_2025): 0 nulls
  race_poc (race_peopleofcolor_pct): 0 nulls
  gender_female (gender_female_pct): 0 nulls

Vulnerability sub-components and score:
cdcr_code  vuln_medical_acuity  vuln_age_over_50  vuln_mental_health  vuln_disability  vuln_race_poc  vuln_gender_female  vulnerability_score
     CHCF             1.000000          1.000000            0.504854         1.000000       0.090454            0.000000             0.599218
      CMF             0.931385          0.810507            0.587379         0.777570       0.175051            0.002445             0.547389
      RJD             0.872935          0.652908            0.618932         0.583178       0.363903            0.000000             0.515309
      SAC             0.869123          0.257036            1.000000        

## 6. Risk score

Risk = 0.25 × Hazard + 0.25 × Exposure + 0.50 × Vulnerability

Normalized 0–100 **cross-period**: current and mid-century scores share the same min/max denominator, so they are directly comparable.

In [8]:
H_cur = df['hazard_current']
H_mid = df['hazard_midcentury']
E = df['exposure_score']
V = df['vulnerability_score']

df['raw_risk_current']    = 0.25 * H_cur + 0.25 * E + 0.50 * V
df['raw_risk_midcentury'] = 0.25 * H_mid + 0.25 * E + 0.50 * V

# Cross-period normalization: min/max taken across both periods together
all_raw = pd.concat([df['raw_risk_current'], df['raw_risk_midcentury']])
raw_min, raw_max = all_raw.min(), all_raw.max()
print(f'Raw risk range (both periods): {raw_min:.4f} – {raw_max:.4f}')

df['risk_score_current']    = (df['raw_risk_current']    - raw_min) / (raw_max - raw_min) * 100
df['risk_score_midcentury'] = (df['raw_risk_midcentury'] - raw_min) / (raw_max - raw_min) * 100

print('\nRisk scores — mid-century ranked:')
print(df[['cdcr_code', 'hazard_midcentury', 'exposure_score', 'vulnerability_score',
          'risk_score_current', 'risk_score_midcentury']]
      .sort_values('risk_score_midcentury', ascending=False)
      .round(2).to_string(index=False))

Raw risk range (both periods): 0.1580 – 0.6011

Risk scores — mid-century ranked:
cdcr_code  hazard_midcentury  exposure_score  vulnerability_score  risk_score_current  risk_score_midcentury
      CIM               1.00            0.52                 0.44               62.00                 100.00
      CMF               0.76            0.49                 0.55               71.17                  96.93
      CIW               0.95            0.54                 0.41               58.74                  94.58
      COR               0.77            0.66                 0.38               56.02                  87.70
     SATF               0.77            0.61                 0.40               55.73                  87.41
      SAC               0.70            0.42                 0.51               58.75                  85.26
      RJD               0.74            0.37                 0.52               60.45                  84.98
     CHCF               0.80            0.09  

## 7. Risk categories and output

Risk categories derived from Jenks natural breaks (k=4) on mid-century risk scores.
Labels: **Lowest → Moderate → High → Highest**.
Same break thresholds applied to current-period scores so both time periods are comparable.

Long format: two rows per facility (current + mid-century).
Saved to `data/cdcr/CDCR_heat_risk_index_additive_25_25_50.csv`.

In [9]:
import os, shutil
import jenkspy

# Risk categories: Jenks natural breaks (k=4) on mid-century risk scores
breaks = jenkspy.jenks_breaks(df['risk_score_midcentury'].tolist(), n_classes=4)
# jenkspy returns [min, b1, b2, b3, max] — use inner breaks + max as bin edges
bin_edges = breaks[1:]  # drop the min
labels = ['Lowest', 'Moderate', 'High', 'Highest']
print(f'Jenks breaks (mid-century, k=4): {[round(b, 2) for b in bin_edges]}')

def risk_label(score, bins=bin_edges, lbls=labels):
    for i, b in enumerate(bins):
        if score <= b:
            return lbls[i]
    return lbls[-1]

df['risk_category_current']    = df['risk_score_current'].apply(risk_label)
df['risk_category_midcentury'] = df['risk_score_midcentury'].apply(risk_label)

print('\nMid-century risk categories:')
print(df[['cdcr_code', 'risk_score_midcentury', 'risk_category_midcentury']]
      .sort_values('risk_score_midcentury', ascending=False).to_string(index=False))

# ── Output ──────────────────────────────────────────────────────────────────
OUT_CSV = 'data/cdcr/CDCR_heat_risk_index_additive_25_25_50.csv'

# Snapshot the v0.1 output before overwriting, so v0.1 ↔ v0.2 stays comparable.
# Idempotent: only snapshots once.
_v01 = 'data/cdcr/CDCR_heat_risk_index_additive_25_25_50_v0.1.csv'
if not os.path.exists(_v01) and os.path.exists(OUT_CSV):
    shutil.copyfile(OUT_CSV, _v01)
    print(f'Snapshotted v0.1 -> {_v01}')

shared_cols = [
    'cdcr_code', 'name', 'latitude', 'longitude', 'average_2025_population',
    'exposure_score', 'vulnerability_score',
    'AQI_norm', 'ratio_indoor_to_outdoor', 'days_indoor_above_78f_2025',
    'uhi_normalized', 'pct_units_refrigeration',
    # vulnerability raw inputs for interpretability
    'medical_acuity', 'cchcs_age_over_50_pct_2025',
    'cchcs_mental_health_eop_pct_2025', 'cchcs_dpp_pct_2025', 'race_peopleofcolor_pct', 'gender_female_pct',
    'rhu_pct_2025',
    # descriptive (not scored)
    'dist_nearest_medical_mi', 'in_urban_area_2020', 'california_model_facility', 'year_opened',
]

current = df[shared_cols + ['hazard_current', 'risk_score_current', 'risk_category_current']].copy()
current = current.rename(columns={
    'hazard_current': 'hazard_score',
    'risk_score_current': 'risk_score',
    'risk_category_current': 'risk_category',
})
current['time_period'] = 'current'

midcentury = df[shared_cols + ['hazard_midcentury', 'risk_score_midcentury', 'risk_category_midcentury']].copy()
midcentury = midcentury.rename(columns={
    'hazard_midcentury': 'hazard_score',
    'risk_score_midcentury': 'risk_score',
    'risk_category_midcentury': 'risk_category',
})
midcentury['time_period'] = 'midcentury'

output = pd.concat([current, midcentury], ignore_index=True)
output = output.sort_values(['cdcr_code', 'time_period']).reset_index(drop=True)

score_cols = ['hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score']
output[score_cols] = output[score_cols].round(2)

# Self-identifying version tag (see analysis/README.md changelog).
output['index_version'] = 'v0.2'

output.to_csv(OUT_CSV, index=False)
print(f'\nSaved {len(output)} rows to {OUT_CSV}')
print(f'Facilities: {output["cdcr_code"].nunique()}, Time periods: {output["time_period"].unique()}, '
      f'version: {output["index_version"].unique()[0]}')
print(f'Columns: {list(output.columns)}')

summary = output[output['time_period'] == 'midcentury'][
    ['cdcr_code', 'hazard_score', 'exposure_score', 'vulnerability_score', 'risk_score', 'risk_category']
].sort_values('risk_score', ascending=False).reset_index(drop=True)
summary.index += 1
print('\nMid-century risk ranking:')
print(summary.to_string())

Jenks breaks (mid-century, k=4): [np.float64(43.66), np.float64(72.05), np.float64(87.7), np.float64(100.0)]

Mid-century risk categories:
cdcr_code  risk_score_midcentury risk_category_midcentury
      CIM             100.000000                  Highest
      CMF              96.930206                  Highest
      CIW              94.576214                  Highest
      COR              87.701254                     High
     SATF              87.414041                     High
      SAC              85.259592                     High
      RJD              84.977678                     High
     CHCF              82.068746                     High
     CCWF              79.546979                     High
      SOL              78.367056                     High
      LAC              78.347461                     High
      VSP              75.241615                     High
      CMC              72.045079                 Moderate
     MCSP              69.549275                 

## 8. PBSP ratio outlier check

In [10]:
# PBSP has ratio_indoor_to_outdoor = 15.75 — all other facilities are < 2.0
# Check how much this outlier inflates PBSP's exposure score vs. a capped version

pbsp = df[df['cdcr_code'] == 'PBSP'].iloc[0]
print(f'PBSP ratio: {pbsp["ratio_indoor_to_outdoor"]}')
print(f'PBSP outdoor 78F days (implied): {pbsp["days_indoor_above_78f_2025"] / pbsp["ratio_indoor_to_outdoor"]:.1f}')
print(f'PBSP exposure_score (with outlier): {pbsp["exposure_score"]:.3f}')
print(f'PBSP exp_ratio (with outlier): {pbsp["exp_ratio"]:.3f}')

# What would PBSP exposure score be if ratio were capped at p95 of other facilities?
other_ratios = df.loc[df['cdcr_code'] != 'PBSP', 'ratio_indoor_to_outdoor']
p95 = other_ratios.quantile(0.95)
print(f'\n95th pctl ratio (excl. PBSP): {p95:.3f}')
print('(No cap applied — outlier retained. Consider sensitivity analysis if PBSP rank is influential.)')

PBSP ratio: 15.75
PBSP outdoor 78F days (implied): 4.0
PBSP exposure_score (with outlier): 0.651
PBSP exp_ratio (with outlier): 1.000

95th pctl ratio (excl. PBSP): 1.699
(No cap applied — outlier retained. Consider sensitivity analysis if PBSP rank is influential.)
